# AUDIT-X v2 · 07 · Family proximity in independent feature spaces

Notebook 06 found that zero-day recall falls as an unseen family gets closer to its nearest known family, but it measured
"closeness" in the deep ensemble's own fine-tuned features. This notebook re-measures proximity in two spaces that **no
evaluated model was trained in**, and re-tests the correlation against the per-family recall the main runs already saved.

| Space | What it is | Why |
|---|---|---|
| **ImageNet** (primary) | Frozen ImageNet ResNet-50, penultimate layer, never fine-tuned on malware | Generic visual similarity, independent of every evaluated model |
| **Histogram** (secondary) | 256-bin grey-level histogram of the raw image (= byte-value distribution of the binary) | Model-free; no learning at all |

For every split, known-family centroids use **training-split images only** (as in notebook 06); an unseen family's centroid
uses all its images. Proximity = highest cosine similarity to any known-family centroid.

**Criterion (fixed before running, same form as notebook 06):** the finding is *confirmed independent of the models* if,
in the ImageNet space, a primary configuration (`agef_sharp`+entropy or `deep_ensemble`+energy) has family-level
Spearman ρ ≤ −0.5 with p < 0.05 on ≥2 datasets.

**Inputs** (Add Input): datasets `malimg-dataset-25`, `melevis`, `dumpware10` **and** the committed outputs of notebooks
**01, 02, 03** (for the splits and the saved per-family recall). Optional: notebook **06**'s output, to print the comparison
with its deep-ensemble-space results. **GPU T4, Internet On.** Commit with *Save & Run All*. Expected: ~20–40 min (estimate).
If one dataset fails to mount, remove it and run the others; the notebook analyses whatever is attached.

In [ ]:
# Cell 1 - inputs: datasets, cached runs, (optional) notebook-06 output
import os, glob, json, re, time, warnings
import numpy as np, pandas as pd, torch
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
OUT = "/kaggle/working/auditx_v2_proximity"; os.makedirs(OUT, exist_ok=True)

DATA = {
    "Malimg":     "/kaggle/input/datasets/mohammadmehedimunna/malimg-dataset-25/Malimg_Dataset_25classes",
    "MaleVis":    "/kaggle/input/datasets/mohammadmehedimunna/melevis/MaleVis",
    "Dumpware10": "/kaggle/input/datasets/mohammadmehedimunna/dumpware10/dumpware10_images",
}
DATA_FOLDER = {"Malimg": "Malimg_Dataset_25classes", "MaleVis": "MaleVis", "Dumpware10": "dumpware10_images"}
RUN_ROOTS = {
    "Malimg":     "/kaggle/input/notebooks/rauf41/01-auditx-malimg",
    "MaleVis":    "/kaggle/input/notebooks/rauf41/02-auditx-malevis-2",
    "Dumpware10": "/kaggle/input/notebooks/rauf41/03-auditx-dumpware10-final",
}
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine but slower)")

def locate(path, folder):
    if os.path.isdir(path): return path
    hits = [p for p in glob.glob(f"/kaggle/input/**/{folder}", recursive=True) if os.path.isdir(p)]
    return hits[0] if hits else None

def locate_run(ds):
    for cand in [os.path.join(RUN_ROOTS[ds], f"auditx_v2_{ds.lower()}", ds)] + \
                glob.glob(f"/kaggle/input/**/auditx_v2_{ds.lower()}/{ds}", recursive=True):
        if os.path.isdir(cand) and "QUICKTEST" not in cand and glob.glob(os.path.join(cand, "seed*", "split.json")):
            return cand
    return None

READY = {}
for ds in DATA:
    dpath, rdir = locate(DATA[ds], DATA_FOLDER[ds]), locate_run(ds)
    pc = os.path.join(os.path.dirname(rdir), "per_class_raw.csv") if rdir else None
    ok = dpath is not None and rdir is not None and os.path.exists(pc)
    print(f"{ds:<11} data={dpath}\n            run={rdir}\n            per_class_raw={'found' if pc and os.path.exists(pc) else 'MISSING'} -> {'READY' if ok else 'SKIPPED'}")
    if ok: READY[ds] = dict(data=dpath, run=rdir, per_class=pc)
ref06 = glob.glob("/kaggle/input/**/auditx_v2_novelty/B_family_summary.csv", recursive=True)
print("notebook-06 summary:", ref06[0] if ref06 else "not attached (comparison will be skipped)")
assert READY, "attach at least one dataset together with its run output"

In [ ]:
# Cell 2 - Dumpware10: rebuild the merged folder exactly as notebook 03 did (identical file names => identical sample order)
def merge_dumpware(raw, merged="/kaggle/working/dumpware10_merged"):
    root = None
    for cand in [os.path.join(raw, "224", "224"), os.path.join(raw, "224"), raw]:
        if os.path.isdir(cand) and {"TRAINING", "TEST"} <= set(os.listdir(cand)):
            root = cand; break
    if root is None:
        print("no TRAINING/TEST split found - using the folder as is"); return raw
    os.makedirs(merged, exist_ok=True)
    for split in ["TRAINING", "TEST"]:
        for cls in sorted(os.listdir(os.path.join(root, split))):
            sd = os.path.join(root, split, cls)
            if not os.path.isdir(sd): continue
            dd = os.path.join(merged, cls); os.makedirs(dd, exist_ok=True)
            for fn in os.listdir(sd):
                dst = os.path.join(dd, f"{split.lower()}__{fn}")
                if not os.path.lexists(dst): os.symlink(os.path.join(sd, fn), dst)
    return merged
if "Dumpware10" in READY:
    READY["Dumpware10"]["data"] = merge_dumpware(READY["Dumpware10"]["data"])
    print("Dumpware10 ->", READY["Dumpware10"]["data"])

In [ ]:
# Cell 3 - verify that sample order matches the cached runs, then extract both feature spaces once per dataset
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

TF = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class Both(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img = Image.open(self.samples[i][0])
        hist = np.asarray(img.convert("L").histogram(), dtype=np.float32)
        return TF(img.convert("RGB")), torch.from_numpy(hist / max(hist.sum(), 1))

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = models.resnet50(weights="IMAGENET1K_V2"); net.fc = torch.nn.Identity(); net = net.to(dev).eval()

FEATS = {}
for ds, R in READY.items():
    base = datasets.ImageFolder(R["data"])
    targets = np.array(base.targets)
    # order check against every cached split
    for sj in sorted(glob.glob(os.path.join(R["run"], "seed*", "split.json"))):
        sp = json.load(open(sj))
        assert sorted(sp["known_names"] + sp["unknown_names"] + sp.get("excluded_classes", [])) == base.classes, f"{ds}: class list differs from the run"
        unk_names = {base.classes[t] for t in targets[sp["unk"]]}
        assert unk_names == set(sp["unknown_names"]), f"{ds} {sj}: sample order differs from the run"
        tr_names = {base.classes[t] for t in targets[sp["train"]]}
        assert tr_names == set(sp["known_names"]), f"{ds} {sj}: training indices do not map to the known classes"
        assert 0 < len(sp["unk"]) <= int(np.isin(targets, [base.classes.index(n) for n in sp["unknown_names"]]).sum())
        assert sp.get("protocol") == "grouped-L1-v3+near-L3-8bit-cap250", f"{ds}: attach the NEW (duplicate-aware) committed versions of notebooks 01-03"
    print(f"{ds}: {len(base)} images, order check OK against {len(glob.glob(os.path.join(R['run'], 'seed*')))} splits")
    cache = os.path.join(OUT, f"features_{ds}.npz")
    if os.path.exists(cache):
        d = np.load(cache); FEATS[ds] = dict(imagenet=d["imagenet"], hist=d["hist"], targets=targets, classes=base.classes); continue
    ld = DataLoader(Both(base.samples), batch_size=64, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())
    fi, fh, t0 = [], [], time.time()
    with torch.no_grad():
        for xb, hb in ld:
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                fi.append(net(xb.to(dev)).float().cpu().numpy())
            fh.append(hb.numpy())
    FEATS[ds] = dict(imagenet=np.concatenate(fi), hist=np.concatenate(fh), targets=targets, classes=base.classes)
    np.savez_compressed(cache, imagenet=FEATS[ds]["imagenet"], hist=FEATS[ds]["hist"])
    print(f"   features extracted in {(time.time()-t0)/60:.1f} min: imagenet {FEATS[ds]['imagenet'].shape}, hist {FEATS[ds]['hist'].shape}")

In [ ]:
# Cell 4 - proximity of each unseen family to its nearest known family, per split and per space
def unit(x): return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
rows = []
for ds, R in READY.items():
    F = FEATS[ds]
    for sj in sorted(glob.glob(os.path.join(R["run"], "seed*", "split.json"))):
        seed = int(re.findall(r"seed(\d+)", sj)[0]); sp = json.load(open(sj))
        train = np.array(sp["train"])
        for space in ("imagenet", "hist"):
            X = unit(F[space])
            known_idx = [F["classes"].index(n) for n in sp["known_names"]]
            cent = unit(np.stack([X[train[F["targets"][train] == c]].mean(0) for c in known_idx]))
            for fam in sp["unknown_names"]:
                c = F["classes"].index(fam)
                fc = unit(X[F["targets"] == c].mean(0, keepdims=True))[0]
                sims = cent @ fc
                rows.append(dict(dataset=ds, seed=seed, space=space, family=fam, sim=float(sims.max()),
                                 nearest=sp["known_names"][int(sims.argmax())]))
P = pd.DataFrame(rows); P.to_csv(f"{OUT}/proximity_raw.csv", index=False)
display(P[P.space == "imagenet"].sort_values(["dataset", "sim"], ascending=[True, False]).round(3))

In [ ]:
# Cell 5 - correlate proximity with the per-family recall saved by the main runs
def spearman_ci(x, y, n_boot=2000, seed=0):
    rho, p = stats.spearmanr(x, y); rng = np.random.RandomState(seed); bs = []
    for _ in range(n_boot):
        i = rng.randint(0, len(x), len(x))
        if len(np.unique(x[i])) > 2: bs.append(stats.spearmanr(x[i], y[i])[0])
    lo, hi = np.nanpercentile(bs, [2.5, 97.5]) if bs else (np.nan, np.nan)
    return rho, p, lo, hi

REC = pd.concat([pd.read_csv(R["per_class"]) for R in READY.values()])
REC = REC[REC.role == "unknown"].rename(columns={"cls": "family", "alarm_rate": "recall"})
J = REC.merge(P, on=["dataset", "seed", "family"])
assert len(J) == len(REC) * 2, "every (split, family, model, score) row should match once per space"
res = []
for (ds, space, m, s), g in J.groupby(["dataset", "space", "model", "score"]):
    fam = g.groupby("family").agg(recall=("recall", "mean"), sim=("sim", "mean"))
    rho, p, lo, hi = spearman_ci(fam.sim.values, fam.recall.values)
    pr, pp = stats.spearmanr(g.sim, g.recall)
    res.append(dict(dataset=ds, space=space, model=m, score=s, n_families=len(fam), n_points=len(g),
                    rho_family=rho, p_family=p, ci_lo=lo, ci_hi=hi, rho_points=pr, p_points=pp))
RES = pd.DataFrame(res); RES.to_csv(f"{OUT}/proximity_vs_recall.csv", index=False)
PRIMARY = [("agef_sharp", "entropy"), ("deep_ensemble", "energy")]
prim = RES[RES.apply(lambda r: (r.model, r.score) in PRIMARY, axis=1)]
print("Primary configurations:"); display(prim.round(3))
print("All configurations, ImageNet space:"); display(RES[RES.space == "imagenet"].round(3))
print("All configurations, histogram space:"); display(RES[RES.space == "hist"].round(3))

# agreement between spaces: do the three proximity measures rank families the same way?
piv = P.groupby(["dataset", "family", "space"]).sim.mean().unstack()
if ref06:
    b06 = pd.read_csv(glob.glob("/kaggle/input/**/auditx_v2_novelty/B_family_raw.csv", recursive=True)[0])
    de = b06[(b06.model == "deep_ensemble") & (b06.score == "energy")].groupby(["dataset", "family"]).sim_ref.mean()
    piv = piv.join(de.rename("deep_ensemble"), how="left")
print("\nSpearman correlation between proximity spaces (family level):")
for ds, g in piv.groupby(level=0):
    print(ds); display(g.corr(method="spearman").round(3))
if ref06:
    r06 = pd.read_csv(ref06[0])
    cmp_ = r06[r06.apply(lambda r: (r.model, r.score) in PRIMARY, axis=1)][["dataset", "model", "score", "rho_family_ref", "p_family_ref"]]
    print("\nNotebook 06 (deep-ensemble space) for comparison:"); display(cmp_.round(3))

In [ ]:
# Cell 6 - figure + verdict
import matplotlib.pyplot as plt
dss = list(READY)
fig, axs = plt.subplots(2, len(dss), figsize=(5 * len(dss), 8), squeeze=False)
for j, ds in enumerate(dss):
    for i, space in enumerate(("imagenet", "hist")):
        a_ = axs[i][j]
        for (m, s), mk in zip(PRIMARY, ["o", "s"]):
            g = J[(J.dataset == ds) & (J.space == space) & (J.model == m) & (J.score == s)]
            fam = g.groupby("family").agg(recall=("recall", "mean"), sim=("sim", "mean"))
            a_.scatter(fam.sim, 100 * fam.recall, marker=mk, alpha=.8, label=f"{m} + {s}")
            if m == "agef_sharp":
                for f, r in fam.iterrows(): a_.annotate(f, (r.sim, 100 * r.recall), fontsize=6, alpha=.7)
        a_.set_title(f"{ds} - {space} space"); a_.set_xlabel("cosine similarity to nearest known family")
        a_.set_ylabel("zero-day recall at 5% FAR (%)"); a_.grid(alpha=.3); a_.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_proximity_independent.pdf"); plt.savefig(f"{OUT}/fig_proximity_independent.png", dpi=200); plt.show()

verdict = {}
for space in ("imagenet", "hist"):
    pr = prim[(prim.space == space)]
    ok = pr[(pr.rho_family <= -0.5) & (pr.p_family < 0.05)]
    verdict[space] = dict(datasets_meeting=sorted(ok.dataset.unique()),
                          holds=ok.dataset.nunique() >= 2,
                          rhos=pr[["dataset", "model", "rho_family", "p_family"]].round(3).values.tolist())
print("=" * 100)
v = verdict["imagenet"]
print(("CONFIRMED" if v["holds"] else "NOT CONFIRMED"), "(primary, ImageNet space): datasets meeting the criterion:", v["datasets_meeting"])
print("   ", v["rhos"])
v = verdict["hist"]
print("Secondary (histogram space): datasets meeting the criterion:", v["datasets_meeting"], "\n   ", v["rhos"])
print("=" * 100)
json.dump(verdict, open(f"{OUT}/verdict_proximity.json", "w"), indent=1, default=str)
print("Outputs:", sorted(os.listdir(OUT)))